# Muse Waffle v3 — Training Notebook

Run the cells below in order: **Setup → Create/Load Model → Train → Test**.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath("."))  # make src/ importable

import torch
from src.dataset import load_cached_paths, download_dataset
from src.model import build_model, load_model, DEFAULT_CONFIG
from src.trainer import train
from src.generator import generate_and_export, show_text_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

ModuleNotFoundError: No module named 'dotenv'

## 1. Create or Load Model

`build_model()` initialises fresh weights and saves `checkpoints/config.json`.  
`load_model()` reloads weights from a previous run — use it after training.  
Edit `cfg` below to change architecture before building.

In [ ]:
from pathlib import Path

WEIGHTS_FILE = Path("checkpoints/model.pt")

if WEIGHTS_FILE.exists():
    print("Found existing weights — loading model from checkpoint.")
    model, cfg = load_model(device=DEVICE)
else:
    print("No checkpoint found — building a new model.")
    cfg = dict(DEFAULT_CONFIG)  # edit this dict to customise the architecture
    # cfg["d_model"]     = 512   # embedding dimension (larger = more capacity)
    # cfg["n_heads"]     = 8     # attention heads (must divide d_model)
    # cfg["n_layers"]    = 6     # transformer blocks
    # cfg["max_seq_len"] = 1024  # context window
    model = build_model(cfg).to(DEVICE)

print(f"Parameters: {model.num_params():,}")
print(f"Config: {cfg}")

## 2. Train

Downloads the dataset if not already cached, then runs the training loop.  
Weights are saved to `checkpoints/model.pt` after every epoch.

In [ ]:
EPOCHS     = 5
BATCH_SIZE = 8
LR         = 3e-4
MAX_FILES  = 50   # number of scores to download/use (increase for better quality)

log_lines = []
def log(msg):
    print(msg)
    log_lines.append(msg)

def on_epoch(epoch, train_loss, val_loss):
    print(f"  → epoch {epoch}  train={train_loss:.4f}  val={val_loss:.4f}")

# Download dataset if not already cached
paths = load_cached_paths()
if len(paths) < MAX_FILES:
    print(f"Downloading up to {MAX_FILES} scores...")
    download_dataset(log, progress=lambda i, n: None, max_files=MAX_FILES)
    paths = load_cached_paths()

print(f"Using {len(paths)} score files.")

stop_flag = [False]  # set stop_flag[0] = True in another cell to interrupt training early
train(
    model=model,
    data_paths=paths[:MAX_FILES],
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    device=DEVICE,
    log=log,
    on_epoch=on_epoch,
    stop_flag=stop_flag,
)

## 3. Test (Generate Music)

Generates a short composition from a text prompt, exports MIDI and MusicXML to `output/`,
and prints a measure-by-measure text preview.

In [ ]:
PROMPT      = "Schubert gentle waltz in G major"
MAX_NEW     = 800   # max music tokens to generate
TEMPERATURE = 0.9   # higher = more random; lower = more conservative
TOP_K       = 50    # restrict sampling to top-k predictions

gen_log = []
def glog(msg):
    print(msg)
    gen_log.append(msg)

paths = generate_and_export(
    prompt=PROMPT,
    max_new=MAX_NEW,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    log=glog,
    device=DEVICE,
)

print(f"\nMIDI:     {paths['midi']}")
print(f"MusicXML: {paths['xml']}")

if paths["xml"]:
    print("\n--- Score preview (first 8 measures) ---")
    show_text_score(paths["xml"], glog)